# 16. Exercise Authoring Prototype

- Mode: manual_preparation
- Goal: create a small `exercise_authoring_spec`, preview draft YAML artifacts, and optionally export drafts for researcher review.
- Docs: `docs_eng/practical_protocols/exercise_authoring_notebook.md` / `docs/practical_protocols/exercise_authoring_notebook.md`
- Inputs: registry YAML files under `data/registries/` and user-selected exercise options.
- Outputs: YAML previews, plus optional draft files under `data/processed/authoring_drafts/<exercise_id>/`.
- Review gate: generated drafts are not canonical files; review `requires_review` before promotion.


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

from movement.exercise_authoring import (
    ExerciseAuthoringSpec,
    artifact_to_yaml,
    derive_movement_pattern_from_authoring_axes,
    draft_artifact_paths,
    generate_authoring_artifacts,
    load_authoring_registries,
    recommend_analysis_templates_for_authoring_axes,
    recommend_camera_positions_for_authoring_axes,
    recommend_counting_templates_for_authoring_axes,
    recommend_phase_templates_for_authoring_axes,
    suggest_body_regions_from_joint_actions,
    write_authoring_draft_artifacts,
)

registries = load_authoring_registries()

## Registry Choices

In [ ]:
choices = {
    "movement_patterns": sorted(registries["movement_patterns"]["patterns"]),
    "support_templates": sorted(registries["support_templates"]["templates"]),
    "phase_templates": sorted(registries["phase_templates"]["templates"]),
    "analysis_templates": sorted(registries["analysis_templates"]["templates"]),
    "performance_templates": sorted(registries["performance_templates"]["templates"]),
    "camera_view_families": list((registries["camera_zones"].get("view_families") or {}).keys()),
    "camera_height_levels": list((registries["camera_zones"].get("height_levels") or {}).keys()),
}
registry_counts = {name: len(values) for name, values in choices.items()}
registry_counts


## Optional Descriptor Selection


In [ ]:
DEFAULT_SPEC_VALUES = {
    "exercise_id": "draft_squat",
    "display_name": "Draft Squat",
    "posture_type": "standing",
    "body_geometry": "neutral_upright",
    "laterality": "bilateral_symmetric",
    "support_template": "bilateral_feet",
    "primary_body_regions": ("hip", "knee", "ankle"),
    "primary_joint_actions": (
        "hip_flexion_extension",
        "knee_flexion_extension",
        "ankle_dorsiflexion_plantarflexion",
    ),
    "secondary_joint_actions": ("trunk_flexion_extension",),
    "primary_plane": "sagittal",
    "secondary_planes": ("frontal", "transverse"),
    "phase_template": "descent_ascent_hip_center",
    "counting_template": "repeated_repetition",
    "camera_view_family": "front_oblique",
    "camera_height_level": "H2",
    "analysis_template": "bilateral_lower_body_closed_chain",
    "target_count_per_set": 10,
}

MOVEMENT_PATTERN_LABELS = {
    "squat": "bilateral lower-body closed-chain bend",
    "lunge": "split-stance lower-body loading",
    "push": "closed-chain upper-body press",
    "anti_rotation": "alternating anti-rotation stability",
}

DISPLAY_LABELS = {
    "support_templates": {
        "bilateral_feet": "two-foot floor support",
        "single_foot": "single-foot floor support",
        "split_stance": "split-stance foot support",
        "hands_and_feet": "hands-and-feet floor support",
        "forearms_and_feet": "forearms-and-feet floor support",
        "hands_and_knees": "hands-and-knees floor support",
        "hands_and_feet_with_trunk": "hands, feet, and trunk floor support",
        "knees_floor": "knees floor support",
        "one_knee_one_foot": "one-knee one-foot floor support",
        "seated_base": "seated base support",
        "supine_body_floor": "supine body floor support",
        "side_body_floor": "side-body floor support",
        "overhead_hang": "overhead hang support",
        "external_object": "external-object support",
    },
    "phase_templates": {
        "descent_ascent_hip_center": "descent/ascent by hip-center trajectory",
        "descent_ascent_nose": "descent/ascent by head trajectory",
        "lift_tap_return_wrist": "lift/tap/return by wrist trajectory",
        "static_hold_center": "static hold by center interval",
        "reach_return_wrist": "reach/return by wrist trajectory",
        "rotate_return_trunk": "rotate/return by trunk rotation",
        "bridge_lift_lower_hip_center": "bridge lift/lower by hip-center trajectory",
    },
    "performance_templates": {
        "repeated_repetition": "same movement repeated",
        "same_side_block_then_switch_5_each": "same-side block then switch",
        "alternating_left_right_pair": "alternating left/right pair",
        "timed_hold_seconds": "timed hold in seconds",
    },
    "analysis_templates": {
        "bilateral_lower_body_closed_chain": "lower-body closed-chain analysis",
        "alternating_lower_body_split_stance": "split-stance lower-body analysis",
        "bilateral_upper_body_inverted_closed_chain": "closed-chain upper-body press analysis",
        "alternating_core_anti_rotation": "anti-rotation control analysis",
    },
}

POSTURE_OPTIONS = [
    ("standing", "standing"),
    ("floor-supported prone", "floor_supported_prone"),
    ("kneeling", "kneeling"),
    ("seated", "seated"),
    ("supine", "supine"),
    ("side-lying", "side_lying"),
    ("hanging", "hanging"),
    ("external-object-supported", "external_object_supported"),
]
BODY_GEOMETRY_LABELS = {
    "neutral_upright": "neutral upright alignment",
    "forward_lean_hinge": "forward-lean hip-hinge alignment",
    "neutral_prone_line": "prone neutral body line",
    "high_hip_inverted_v": "high-hip inverted-V alignment",
    "quadruped": "hands-and-knees quadruped alignment",
    "tall_kneeling_upright": "tall-kneeling upright alignment",
    "half_kneeling_upright": "half-kneeling upright alignment",
    "seated_flexed_trunk": "seated flexed-trunk alignment",
    "long_sitting": "long-sitting alignment",
    "cross_legged_sitting": "cross-legged sitting alignment",
    "supine_hooklying": "supine hooklying alignment",
    "supine_tabletop": "supine tabletop-limb alignment",
    "supine_straight_leg": "supine straight-leg alignment",
    "supine_hollow": "supine hollow-body alignment",
    "side_supported": "side-supported neutral alignment",
    "side_plank_line": "side-plank line alignment",
    "side_lying_relaxed": "side-lying relaxed alignment",
    "hanging_tuck": "hanging tuck alignment",
    "hanging_pike": "hanging pike alignment",
    "supported_incline_line": "externally supported incline line",
    "machine_supported_fixed_trunk": "machine-supported fixed trunk",
    "custom": "custom geometry",
}
POSTURE_BODY_GEOMETRY_VALUES = {
    "standing": ("neutral_upright", "forward_lean_hinge"),
    "floor_supported_prone": ("neutral_prone_line", "high_hip_inverted_v", "quadruped"),
    "kneeling": ("neutral_upright", "tall_kneeling_upright", "half_kneeling_upright", "quadruped"),
    "seated": ("neutral_upright", "seated_flexed_trunk", "long_sitting", "cross_legged_sitting"),
    "supine": ("supine_hooklying", "supine_tabletop", "supine_straight_leg", "supine_hollow"),
    "side_lying": ("side_supported", "side_plank_line", "side_lying_relaxed"),
    "hanging": ("neutral_upright", "hanging_tuck", "hanging_pike"),
    "external_object_supported": ("neutral_upright", "neutral_prone_line", "side_supported", "supported_incline_line", "machine_supported_fixed_trunk", "custom"),
}
POSTURE_SUPPORT_TEMPLATE_VALUES = {
    "standing": ("bilateral_feet", "split_stance", "single_foot"),
    "floor_supported_prone": ("hands_and_feet", "forearms_and_feet", "hands_and_knees", "hands_and_feet_with_trunk"),
    "kneeling": ("knees_floor", "one_knee_one_foot", "hands_and_knees"),
    "seated": ("seated_base",),
    "supine": ("supine_body_floor",),
    "side_lying": ("side_body_floor",),
    "hanging": ("overhead_hang",),
    "external_object_supported": ("external_object", "bilateral_feet", "split_stance", "hands_and_feet"),
}
LATERALITY_OPTIONS = [
    ("bilateral symmetric", "bilateral_symmetric"),
    ("bilateral asymmetric", "bilateral_asymmetric"),
    ("alternating sides", "alternating"),
    ("unilateral left", "unilateral_left"),
    ("unilateral right", "unilateral_right"),
    ("unilateral unspecified", "unilateral_unspecified"),
]
BODY_REGION_OPTIONS = [
    ("hip", "hip"),
    ("knee", "knee"),
    ("ankle", "ankle"),
    ("shoulder", "shoulder"),
    ("elbow", "elbow"),
    ("wrist", "wrist"),
    ("trunk", "trunk"),
    ("pelvis", "pelvis"),
    ("foot", "foot"),
    ("hand", "hand"),
]
JOINT_ACTION_OPTIONS = [
    ("hip flexion/extension", "hip_flexion_extension"),
    ("knee flexion/extension", "knee_flexion_extension"),
    ("ankle dorsiflexion/plantarflexion", "ankle_dorsiflexion_plantarflexion"),
    ("shoulder flexion/extension", "shoulder_flexion_extension"),
    ("elbow flexion/extension", "elbow_flexion_extension"),
    ("wrist flexion/extension", "wrist_flexion_extension"),
    ("trunk flexion/extension", "trunk_flexion_extension"),
    ("trunk lateral flexion", "trunk_lateral_flexion"),
    ("trunk rotation", "trunk_rotation_proxy"),
    ("pelvis lateral tilt", "pelvis_lateral_tilt_proxy"),
    ("pelvis rotation", "pelvis_rotation_proxy"),
    ("pelvis anterior/posterior tilt", "pelvis_anterior_posterior_tilt_proxy"),
    ("scapular stability", "scapular_stability_proxy"),
    ("anti-rotation control", "anti_rotation_control"),
    ("weight-shift control", "weight_shift_control"),
]
ANATOMICAL_PLANE_OPTIONS = [
    ("sagittal", "sagittal"),
    ("frontal", "frontal"),
    ("transverse", "transverse"),
]
PLANE_OPTIONS = ANATOMICAL_PLANE_OPTIONS + [("static", "static")]


def labeled_options(group, values):
    labels = DISPLAY_LABELS.get(group, {})
    return [(labels.get(value, value.replace("_", " ")), value) for value in values]


def phase_template_options(recommended_templates=()):
    labels = DISPLAY_LABELS.get("phase_templates", {})
    templates = registries["phase_templates"].get("templates") or {}
    recommended = tuple(
        value for value in recommended_templates if value in choices["phase_templates"]
    )
    ordered_values = list(recommended) + [
        value for value in choices["phase_templates"] if value not in recommended
    ]
    options = []
    for value in ordered_values:
        label = labels.get(value, value.replace("_", " "))
        markers = []
        if value in recommended:
            markers.append("recommended")
        status = templates.get(value, {}).get("implementation_status", "implemented")
        if status != "implemented":
            markers.append(status)
        if markers:
            label = f"{label} [{' / '.join(markers)}]"
        options.append((label, value))
    return options


def phase_template_status_html(phase_template, recommended_templates=()):
    templates = registries["phase_templates"].get("templates") or {}
    template = templates.get(phase_template, {})
    recommended = tuple(recommended_templates)
    messages = []
    if recommended:
        if phase_template in recommended:
            messages.append("<span style='color:#047857'>Recommended for the current selections.</span>")
        else:
            joined = ", ".join(recommended)
            messages.append(
                "<span style='color:#92400e'><b>Note:</b> "
                f"This phase is not currently recommended. Recommended: {joined}.</span>"
            )
    else:
        messages.append("<span style='color:#6b7280'>No strong phase recommendation for the current selections.</span>")

    status = template.get("implementation_status", "implemented")
    if status != "implemented":
        message = template.get("authoring_warning") or "This phase template requires review before canonical use."
        messages.append(f"<span style='color:#b45309'><b>Warning:</b> {message}</span>")
    return "<br>".join(messages)


def counting_template_options(recommended_templates=()):
    labels = DISPLAY_LABELS.get("performance_templates", {})
    templates = registries["performance_templates"].get("templates") or {}
    recommended = tuple(
        value for value in recommended_templates if value in choices["performance_templates"]
    )
    ordered_values = list(recommended) + [
        value for value in choices["performance_templates"] if value not in recommended
    ]
    options = []
    for value in ordered_values:
        label = labels.get(value, value.replace("_", " "))
        markers = []
        if value in recommended:
            markers.append("recommended")
        status = templates.get(value, {}).get("implementation_status", "implemented")
        if status != "implemented":
            markers.append(status)
        if markers:
            label = f"{label} [{' / '.join(markers)}]"
        options.append((label, value))
    return options


def counting_template_status_html(counting_template, recommended_templates=()):
    templates = registries["performance_templates"].get("templates") or {}
    template = templates.get(counting_template, {})
    recommended = tuple(recommended_templates)
    messages = []
    if recommended:
        if counting_template in recommended:
            messages.append("<span style='color:#047857'>Recommended for the current selections.</span>")
        else:
            joined = ", ".join(recommended)
            messages.append(
                "<span style='color:#92400e'><b>Note:</b> "
                f"This counting template is not currently recommended. Recommended: {joined}.</span>"
            )
    else:
        messages.append("<span style='color:#6b7280'>No strong counting recommendation for the current selections.</span>")

    status = template.get("implementation_status", "implemented")
    if status != "implemented":
        message = template.get("authoring_warning") or "This counting template requires review before canonical use."
        messages.append(f"<span style='color:#b45309'><b>Warning:</b> {message}</span>")
    return "<br>".join(messages)


def analysis_template_options(recommended_templates=()):
    labels = DISPLAY_LABELS.get("analysis_templates", {})
    recommended = tuple(
        value for value in recommended_templates if value in choices["analysis_templates"]
    )
    ordered_values = list(recommended) + [
        value for value in choices["analysis_templates"] if value not in recommended
    ]
    options = []
    for value in ordered_values:
        label = labels.get(value, value.replace("_", " "))
        if value in recommended:
            label = f"{label} [recommended]"
        options.append((label, value))
    return options


def analysis_template_status_html(analysis_template, recommended_templates=()):
    recommended = tuple(recommended_templates)
    if recommended:
        if analysis_template in recommended:
            return "<span style='color:#047857'>Recommended for the current selections.</span>"
        joined = ", ".join(recommended)
        return (
            "<span style='color:#92400e'><b>Note:</b> "
            f"This analysis family is not currently recommended. Recommended: {joined}.</span>"
        )
    return "<span style='color:#6b7280'>No strong analysis-family recommendation for the current selections.</span>"



def camera_position_id(view_family, height_level):
    return f"{view_family}/{height_level}"


def split_camera_position(position_id):
    return tuple(position_id.split("/", maxsplit=1))


def camera_view_family_label(view_family):
    family_info = (registries["camera_zones"].get("view_families") or {}).get(view_family, {})
    label = family_info.get("label", view_family)
    member_zones = family_info.get("member_zones") or []
    zone_text = "/".join(member_zones)
    plane = family_info.get("observation_plane")
    suffix = f" ({zone_text})" if zone_text else ""
    plane_suffix = f" - {plane}" if plane else ""
    return f"{label}{suffix}{plane_suffix}"


def camera_height_label(height_level):
    height_info = (registries["camera_zones"].get("height_levels") or {}).get(height_level, {})
    label = height_info.get("label", height_level)
    height_cm = height_info.get("height_cm")
    if isinstance(height_cm, list) and len(height_cm) == 2:
        return f"{height_level} {label} ({height_cm[0]}-{height_cm[1]} cm)"
    return f"{height_level} {label}"


def all_camera_positions():
    return tuple(
        camera_position_id(view_family, height_level)
        for height_level in choices["camera_height_levels"]
        for view_family in choices["camera_view_families"]
    )


def _camera_position_parts(position_ids):
    return [split_camera_position(position_id) for position_id in position_ids]


def camera_view_family_options(recommended_positions=()):
    recommended_views = tuple(
        dict.fromkeys(view for view, _height in _camera_position_parts(recommended_positions))
    )
    ordered = list(recommended_views) + [
        view for view in choices["camera_view_families"] if view not in recommended_views
    ]
    options = []
    for view_family in ordered:
        marker = " [recommended]" if view_family in recommended_views else ""
        options.append((f"{camera_view_family_label(view_family)}{marker}", view_family))
    return options


def camera_height_options(recommended_positions=()):
    recommended_heights = tuple(
        dict.fromkeys(height for _view, height in _camera_position_parts(recommended_positions))
    )
    ordered = list(recommended_heights) + [
        height for height in choices["camera_height_levels"] if height not in recommended_heights
    ]
    options = []
    for height in ordered:
        marker = " [recommended]" if height in recommended_heights else ""
        options.append((f"{camera_height_label(height)}{marker}", height))
    return options


def camera_position_status_html(view_family, height_level, recommended_positions=()):
    position_id = camera_position_id(view_family, height_level)
    recommended = tuple(recommended_positions)
    messages = [
        f"<span><b>Selected view:</b> {camera_view_family_label(view_family)}</span>",
        f"<span><b>Selected height:</b> {camera_height_label(height_level)}</span>",
    ]
    if recommended:
        if position_id in recommended:
            messages.append("<span style='color:#047857'>Recommended for the current selections.</span>")
        else:
            joined = ", ".join(recommended)
            messages.append(
                "<span style='color:#92400e'><b>Note:</b> "
                f"This view/H position is not currently recommended. Recommended: {joined}. "
                "It can still be exported as non-recommended provenance.</span>"
            )
    else:
        messages.append("<span style='color:#6b7280'>No strong camera-position recommendation for the current selections.</span>")
    return "<br>".join(messages)


def body_geometry_options_for_posture(posture_type):
    values = POSTURE_BODY_GEOMETRY_VALUES.get(posture_type, ())
    return [(BODY_GEOMETRY_LABELS[value], value) for value in values]


def support_template_options_for_posture(posture_type):
    values = POSTURE_SUPPORT_TEMPLATE_VALUES.get(posture_type, ())
    return labeled_options("support_templates", values)


def secondary_plane_options_for_primary(primary_plane):
    if primary_plane == "static":
        return ANATOMICAL_PLANE_OPTIONS
    return [
        (label, value)
        for label, value in ANATOMICAL_PLANE_OPTIONS
        if value != primary_plane
    ]


def with_derived_movement_pattern(raw_values):
    values = dict(raw_values)
    values["primary_body_regions"] = tuple(values.get("primary_body_regions") or ())
    values["primary_joint_actions"] = tuple(values.get("primary_joint_actions") or ())
    values["secondary_joint_actions"] = tuple(values.get("secondary_joint_actions") or ())
    values["secondary_planes"] = tuple(values.get("secondary_planes") or ())
    values["movement_pattern"] = derive_movement_pattern_from_authoring_axes(
        posture_type=values["posture_type"],
        body_geometry=values["body_geometry"],
        laterality=values["laterality"],
        support_template=values["support_template"],
        primary_body_regions=values["primary_body_regions"],
        primary_joint_actions=values["primary_joint_actions"],
        secondary_joint_actions=values["secondary_joint_actions"],
    )
    values["movement_pattern_source"] = "derived_from_joint_actions_and_context"
    return values


widget_spec_values = with_derived_movement_pattern(DEFAULT_SPEC_VALUES)

try:
    import ipywidgets as widgets
    from IPython.display import clear_output, display
except ImportError:
    widgets = None
    print("ipywidgets is not available. Edit DEFAULT_SPEC_VALUES directly, then run the next cell.")

if widgets is not None:
    def _checkbox_group(options, selected_values):
        selected = set(selected_values)
        boxes = [
            widgets.Checkbox(value=value in selected, description=label, indent=False)
            for label, value in options
        ]
        return boxes, widgets.VBox(boxes)

    def _checkbox_row(options, selected_values):
        selected = set(selected_values)
        boxes = [
            widgets.Checkbox(value=value in selected, description=label, indent=False)
            for label, value in options
        ]
        return boxes, widgets.HBox(boxes)

    def _selected_checkbox_values(boxes, options):
        return tuple(
            value
            for box, (_label, value) in zip(boxes, options)
            if box.value
        )

    exercise_id_input = widgets.Text(
        value=DEFAULT_SPEC_VALUES["exercise_id"],
        description="exercise_id",
        layout=widgets.Layout(width="420px"),
    )
    display_name_input = widgets.Text(
        value=DEFAULT_SPEC_VALUES["display_name"],
        description="display",
        layout=widgets.Layout(width="420px"),
    )
    posture_dd = widgets.Dropdown(
        options=POSTURE_OPTIONS,
        value=DEFAULT_SPEC_VALUES["posture_type"],
        description="posture",
    )
    body_geometry_dd = widgets.Dropdown(
        options=body_geometry_options_for_posture(DEFAULT_SPEC_VALUES["posture_type"]),
        value=DEFAULT_SPEC_VALUES["body_geometry"],
        description="geometry",
    )
    laterality_dd = widgets.Dropdown(
        options=LATERALITY_OPTIONS,
        value=DEFAULT_SPEC_VALUES["laterality"],
        description="laterality",
    )
    support_dd = widgets.Dropdown(
        options=support_template_options_for_posture(DEFAULT_SPEC_VALUES["posture_type"]),
        value=DEFAULT_SPEC_VALUES["support_template"],
        description="support",
    )

    def _sync_posture_dependent_options(change=None):
        geometry_options = body_geometry_options_for_posture(posture_dd.value)
        allowed_geometries = {value for _label, value in geometry_options}
        body_geometry_dd.options = geometry_options
        if body_geometry_dd.value not in allowed_geometries:
            body_geometry_dd.value = geometry_options[0][1]

        support_options = support_template_options_for_posture(posture_dd.value)
        allowed_supports = {value for _label, value in support_options}
        support_dd.options = support_options
        if support_dd.value not in allowed_supports:
            support_dd.value = support_options[0][1]

    posture_dd.observe(_sync_posture_dependent_options, names="value")
    primary_action_boxes, primary_actions_ui = _checkbox_group(
        JOINT_ACTION_OPTIONS,
        DEFAULT_SPEC_VALUES["primary_joint_actions"],
    )
    secondary_action_boxes, secondary_actions_ui = _checkbox_group(
        JOINT_ACTION_OPTIONS,
        DEFAULT_SPEC_VALUES["secondary_joint_actions"],
    )
    suggested_initial_regions = suggest_body_regions_from_joint_actions(
        DEFAULT_SPEC_VALUES["primary_joint_actions"],
        DEFAULT_SPEC_VALUES["secondary_joint_actions"],
    ) or DEFAULT_SPEC_VALUES["primary_body_regions"]
    region_boxes, regions_ui = _checkbox_group(
        BODY_REGION_OPTIONS,
        suggested_initial_regions,
    )
    region_sync_state = {"manual_override": False, "syncing": False}
    apply_region_suggestion_button = widgets.Button(
        description="Use suggested regions",
        button_style="info",
    )
    region_status = widgets.HTML()

    def _selected_region_values():
        return _selected_checkbox_values(region_boxes, BODY_REGION_OPTIONS)

    def _current_suggested_regions():
        return suggest_body_regions_from_joint_actions(
            _selected_checkbox_values(primary_action_boxes, JOINT_ACTION_OPTIONS),
            _selected_checkbox_values(secondary_action_boxes, JOINT_ACTION_OPTIONS),
        )

    def _set_region_checkboxes(values):
        selected = set(values)
        region_sync_state["syncing"] = True
        for box, (_label, value) in zip(region_boxes, BODY_REGION_OPTIONS):
            box.value = value in selected
        region_sync_state["syncing"] = False

    def _refresh_region_status():
        suggested = _current_suggested_regions()
        mode = "manual override" if region_sync_state["manual_override"] else "suggested"
        region_status.value = f"<span>{mode}: {', '.join(suggested) or 'none'}</span>"

    def _mark_regions_manual(change=None):
        if not region_sync_state["syncing"]:
            region_sync_state["manual_override"] = True
            _refresh_region_status()

    def _sync_regions_from_actions(change=None):
        if region_sync_state["manual_override"]:
            _refresh_region_status()
            return
        _set_region_checkboxes(_current_suggested_regions())
        _refresh_region_status()

    def _use_suggested_regions(_):
        region_sync_state["manual_override"] = False
        _set_region_checkboxes(_current_suggested_regions())
        _refresh_region_status()

    for box in primary_action_boxes + secondary_action_boxes:
        box.observe(_sync_regions_from_actions, names="value")
    for box in region_boxes:
        box.observe(_mark_regions_manual, names="value")
    apply_region_suggestion_button.on_click(_use_suggested_regions)
    _sync_regions_from_actions()
    primary_plane_toggle = widgets.ToggleButtons(
        options=PLANE_OPTIONS,
        value=DEFAULT_SPEC_VALUES["primary_plane"],
        description="",
        style={"button_width": "96px"},
    )
    secondary_planes_ui = widgets.HBox(
        layout=widgets.Layout(margin="0 0 0 16px"),
    )
    secondary_plane_state = {"boxes": [], "options": ()}

    def _sync_secondary_plane_options(change=None):
        if secondary_plane_state["boxes"]:
            selected = set(
                _selected_checkbox_values(
                    secondary_plane_state["boxes"],
                    secondary_plane_state["options"],
                )
            )
        else:
            selected = set(DEFAULT_SPEC_VALUES["secondary_planes"])
        selected.discard(primary_plane_toggle.value)
        options = secondary_plane_options_for_primary(primary_plane_toggle.value)
        boxes, row = _checkbox_row(options, selected)
        secondary_plane_state["boxes"] = boxes
        secondary_plane_state["options"] = options
        secondary_planes_ui.children = row.children

    primary_plane_toggle.observe(_sync_secondary_plane_options, names="value")
    _sync_secondary_plane_options()
    def _current_phase_recommendations():
        return recommend_phase_templates_for_authoring_axes(
            posture_type=posture_dd.value,
            body_geometry=body_geometry_dd.value,
            support_template=support_dd.value,
            primary_joint_actions=_selected_checkbox_values(
                primary_action_boxes,
                JOINT_ACTION_OPTIONS,
            ),
            secondary_joint_actions=_selected_checkbox_values(
                secondary_action_boxes,
                JOINT_ACTION_OPTIONS,
            ),
            primary_plane=primary_plane_toggle.value,
        )

    phase_recommendation_state = {"values": _current_phase_recommendations()}
    phase_dd = widgets.Dropdown(
        options=phase_template_options(phase_recommendation_state["values"]),
        value=DEFAULT_SPEC_VALUES["phase_template"],
        description="phase",
    )
    phase_warning = widgets.HTML()

    def _sync_phase_warning(change=None):
        phase_warning.value = phase_template_status_html(
            phase_dd.value,
            phase_recommendation_state["values"],
        )

    def _sync_phase_options(change=None):
        current_value = phase_dd.value
        recommendations = _current_phase_recommendations()
        phase_recommendation_state["values"] = recommendations
        phase_dd.options = phase_template_options(recommendations)
        option_values = {value for _label, value in phase_dd.options}
        if current_value in option_values:
            phase_dd.value = current_value
        elif recommendations:
            phase_dd.value = recommendations[0]
        else:
            phase_dd.value = phase_dd.options[0][1]
        _sync_phase_warning()

    phase_dd.observe(_sync_phase_warning, names="value")
    for widget in (posture_dd, body_geometry_dd, support_dd, primary_plane_toggle):
        widget.observe(_sync_phase_options, names="value")
    for box in primary_action_boxes + secondary_action_boxes:
        box.observe(_sync_phase_options, names="value")
    _sync_phase_options()
    def _current_counting_recommendations():
        return recommend_counting_templates_for_authoring_axes(
            laterality=laterality_dd.value,
            phase_template=phase_dd.value,
        )

    counting_recommendation_state = {"values": _current_counting_recommendations()}
    counting_dd = widgets.Dropdown(
        options=counting_template_options(counting_recommendation_state["values"]),
        value=DEFAULT_SPEC_VALUES["counting_template"],
        description="counting",
    )
    counting_warning = widgets.HTML()

    def _sync_counting_warning(change=None):
        counting_warning.value = counting_template_status_html(
            counting_dd.value,
            counting_recommendation_state["values"],
        )

    def _sync_counting_options(change=None):
        current_value = counting_dd.value
        recommendations = _current_counting_recommendations()
        counting_recommendation_state["values"] = recommendations
        counting_dd.options = counting_template_options(recommendations)
        option_values = {value for _label, value in counting_dd.options}
        if current_value in option_values:
            counting_dd.value = current_value
        elif recommendations:
            counting_dd.value = recommendations[0]
        else:
            counting_dd.value = counting_dd.options[0][1]
        _sync_counting_warning()

    counting_dd.observe(_sync_counting_warning, names="value")
    laterality_dd.observe(_sync_counting_options, names="value")
    phase_dd.observe(_sync_counting_options, names="value")
    _sync_counting_options()
    def _current_camera_recommendations():
        return recommend_camera_positions_for_authoring_axes(
            posture_type=posture_dd.value,
            laterality=laterality_dd.value,
            support_template=support_dd.value,
            primary_body_regions=_selected_region_values(),
            primary_joint_actions=_selected_checkbox_values(
                primary_action_boxes,
                JOINT_ACTION_OPTIONS,
            ),
            secondary_joint_actions=_selected_checkbox_values(
                secondary_action_boxes,
                JOINT_ACTION_OPTIONS,
            ),
            primary_plane=primary_plane_toggle.value,
        )

    camera_recommendation_state = {"values": _current_camera_recommendations()}
    camera_view_dd = widgets.Dropdown(
        options=camera_view_family_options(camera_recommendation_state["values"]),
        value=DEFAULT_SPEC_VALUES["camera_view_family"],
        description="camera view",
        layout=widgets.Layout(width="420px"),
    )
    camera_height_dd = widgets.Dropdown(
        options=camera_height_options(camera_recommendation_state["values"]),
        value=DEFAULT_SPEC_VALUES["camera_height_level"],
        description="height H",
        layout=widgets.Layout(width="300px"),
    )
    camera_warning = widgets.HTML()

    def _sync_camera_warning(change=None):
        camera_warning.value = camera_position_status_html(
            camera_view_dd.value,
            camera_height_dd.value,
            camera_recommendation_state["values"],
        )

    def _sync_camera_options(change=None):
        current_view = camera_view_dd.value
        current_height = camera_height_dd.value
        recommendations = _current_camera_recommendations()
        camera_recommendation_state["values"] = recommendations
        camera_view_dd.options = camera_view_family_options(recommendations)
        camera_height_dd.options = camera_height_options(recommendations)
        view_values = {value for _label, value in camera_view_dd.options}
        height_values = {value for _label, value in camera_height_dd.options}
        camera_view_dd.value = current_view if current_view in view_values else camera_view_dd.options[0][1]
        camera_height_dd.value = current_height if current_height in height_values else camera_height_dd.options[0][1]
        _sync_camera_warning()

    camera_view_dd.observe(_sync_camera_warning, names="value")
    camera_height_dd.observe(_sync_camera_warning, names="value")
    for widget in (posture_dd, laterality_dd, support_dd, primary_plane_toggle):
        widget.observe(_sync_camera_options, names="value")
    for box in primary_action_boxes + secondary_action_boxes + region_boxes:
        box.observe(_sync_camera_options, names="value")
    _sync_camera_options()
    def _current_analysis_recommendations():
        return recommend_analysis_templates_for_authoring_axes(
            posture_type=posture_dd.value,
            body_geometry=body_geometry_dd.value,
            laterality=laterality_dd.value,
            support_template=support_dd.value,
            primary_body_regions=_selected_region_values(),
            primary_joint_actions=_selected_checkbox_values(
                primary_action_boxes,
                JOINT_ACTION_OPTIONS,
            ),
            secondary_joint_actions=_selected_checkbox_values(
                secondary_action_boxes,
                JOINT_ACTION_OPTIONS,
            ),
        )

    analysis_recommendation_state = {"values": _current_analysis_recommendations()}
    analysis_dd = widgets.Dropdown(
        options=analysis_template_options(analysis_recommendation_state["values"]),
        value=DEFAULT_SPEC_VALUES["analysis_template"],
        description="analysis",
        layout=widgets.Layout(width="520px"),
    )
    analysis_warning = widgets.HTML()

    def _sync_analysis_warning(change=None):
        analysis_warning.value = analysis_template_status_html(
            analysis_dd.value,
            analysis_recommendation_state["values"],
        )

    def _sync_analysis_options(change=None):
        current_value = analysis_dd.value
        recommendations = _current_analysis_recommendations()
        analysis_recommendation_state["values"] = recommendations
        analysis_dd.options = analysis_template_options(recommendations)
        option_values = {value for _label, value in analysis_dd.options}
        if current_value in option_values:
            analysis_dd.value = current_value
        elif recommendations:
            analysis_dd.value = recommendations[0]
        else:
            analysis_dd.value = analysis_dd.options[0][1]
        _sync_analysis_warning()

    analysis_dd.observe(_sync_analysis_warning, names="value")
    for widget in (posture_dd, body_geometry_dd, laterality_dd, support_dd):
        widget.observe(_sync_analysis_options, names="value")
    for box in primary_action_boxes + secondary_action_boxes + region_boxes:
        box.observe(_sync_analysis_options, names="value")
    _sync_analysis_options()
    target_count_input = widgets.BoundedIntText(
        value=DEFAULT_SPEC_VALUES["target_count_per_set"],
        min=1,
        max=200,
        description="count/set",
    )

    apply_button = widgets.Button(description="Use selections", button_style="success")
    selection_output = widgets.Output()

    def _collect_widget_axis_values():
        return {
            "exercise_id": exercise_id_input.value.strip(),
            "display_name": display_name_input.value.strip(),
            "posture_type": posture_dd.value,
            "body_geometry": body_geometry_dd.value,
            "laterality": laterality_dd.value,
            "support_template": support_dd.value,
            "primary_body_regions": _selected_region_values(),
            "primary_joint_actions": _selected_checkbox_values(
                primary_action_boxes,
                JOINT_ACTION_OPTIONS,
            ),
            "secondary_joint_actions": _selected_checkbox_values(
                secondary_action_boxes,
                JOINT_ACTION_OPTIONS,
            ),
            "primary_plane": primary_plane_toggle.value,
            "secondary_planes": _selected_checkbox_values(
                secondary_plane_state["boxes"],
                secondary_plane_state["options"],
            ),
            "phase_template": phase_dd.value,
            "counting_template": counting_dd.value,
            "camera_view_family": camera_view_dd.value,
            "camera_height_level": camera_height_dd.value,
            "analysis_template": analysis_dd.value,
            "target_count_per_set": target_count_input.value,
        }

    def _use_selections(_):
        global widget_spec_values
        with selection_output:
            clear_output()
            try:
                widget_spec_values = with_derived_movement_pattern(_collect_widget_axis_values())
            except ValueError as exc:
                print(exc)
                return
            derived = widget_spec_values["movement_pattern"]
            print("widget_spec_values updated:")
            source = widget_spec_values["movement_pattern_source"]
            print(f"  movement_pattern: {MOVEMENT_PATTERN_LABELS.get(derived, derived)} ({derived}; {source})")
            for key, value in widget_spec_values.items():
                if key != "movement_pattern":
                    print(f"  {key}: {value}")

    apply_button.on_click(_use_selections)

    action_region_ui = widgets.HBox(
        [
            widgets.VBox([widgets.HTML("<b>Primary joint actions</b>"), primary_actions_ui]),
            widgets.VBox([widgets.HTML("<b>Secondary joint actions</b>"), secondary_actions_ui]),
            widgets.VBox([
                widgets.HTML("<b>Analysis focus regions</b>"),
                regions_ui,
                widgets.HBox([apply_region_suggestion_button, region_status]),
            ]),
        ],
        layout=widgets.Layout(align_items="flex-start"),
    )
    plane_ui = widgets.VBox([
        widgets.HBox([widgets.HTML("<b>Primary plane</b>"), primary_plane_toggle]),
        widgets.HBox([widgets.HTML("<b>Secondary planes</b>"), secondary_planes_ui]),
    ])

    display(widgets.VBox([
        widgets.HBox([exercise_id_input, display_name_input]),
        widgets.HBox([posture_dd, body_geometry_dd]),
        widgets.HBox([laterality_dd, support_dd]),
        action_region_ui,
        plane_ui,
        widgets.VBox([widgets.HBox([phase_dd, counting_dd]), phase_warning, counting_warning]),
        widgets.HBox([target_count_input]),
        widgets.VBox([widgets.HBox([camera_view_dd, camera_height_dd]), camera_warning]),
        widgets.VBox([analysis_dd, analysis_warning]),
        apply_button,
        selection_output,
    ]))


## Authoring Spec

In [ ]:
# If you used the descriptor cell above, click "Use selections" before running this cell.
# Without widgets, edit DEFAULT_SPEC_VALUES in the previous cell.
spec_values = with_derived_movement_pattern(widget_spec_values)
spec = ExerciseAuthoringSpec.from_mapping(spec_values)
spec.as_dict()


## YAML Preview

In [ ]:
artifacts = generate_authoring_artifacts(spec, registries)
artifacts.keys()

In [ ]:
print(artifact_to_yaml(artifacts["exercise_definition"]))

In [ ]:
print(artifact_to_yaml(artifacts["analysis_profile"]))

In [ ]:
print(artifact_to_yaml(artifacts["performance_protocol"]))

In [ ]:
print(artifact_to_yaml(artifacts["camera_protocol"]))

## Review Checklist

In [ ]:
review_items = {name: artifact["requires_review"] for name, artifact in artifacts.items()}
review_items

## Draft Write

In [ ]:
# Set WRITE_DRAFTS=True only after reviewing the YAML previews.
# Set OVERWRITE_EXISTING=True only when intentionally replacing a prior draft.
WRITE_DRAFTS = True
OVERWRITE_EXISTING = False

if WRITE_DRAFTS:
    paths = write_authoring_draft_artifacts(
        artifacts,
        overwrite=OVERWRITE_EXISTING,
    )
else:
    paths = draft_artifact_paths(spec.exercise_id)
    print("Dry run only. Set WRITE_DRAFTS=True to export draft YAML files.")

paths
